#### Test notebook for the data pipeline

##### Going through step by step

In [4]:
# set paths and import library

import csv
import time

# Full path to the input text file
input_path = r"C:\Users\tcolw\Documents\local_coding_projs\for_fun_weekly_projects\12_py_pipeline\BDSexp2602.txt"

# Where we want the CSV to be saved
output_path = r"C:\Users\tcolw\Documents\local_coding_projs\for_fun_weekly_projects\12_py_pipeline\BDSexp2602_parsed.csv"

In [2]:
# Each tuple: (column_name, start_position, end_position)
# Positions are 1-based and inclusive, as in the UK Trade Info table.
FIELDS = [
    ("PERREF",            1,   6),  # Period Reference (YYYYMM)
    ("TYPE",              7,   7),  # Record type
    ("MONTHAC",           8,  13),  # Month of Account
    ("COMCODE",          14,  21),  # Commodity code (8 digits)
    ("SITC",             22,  26),  # SITC code (5 digits)
    ("COD_SEQ",          27,  29),  # Country code (numeric)
    ("COD_ALPHA",        30,  31),  # Country code (alpha)
    ("PORT_SEQ",         32,  34),  # Port code (numeric)
    ("PORT_ALPHA",       35,  37),  # Port code (alpha)
    ("COO_SEQ",          38,  40),  # Country of Origin (numeric)
    ("COO_ALPHA",        41,  42),  # Country of Origin (alpha)
    ("MODE_OF_TRANSPORT",43,  44),  # Mode of transport
    ("STAT_VALUE",       45,  56),  # Statistical value
    ("NET_MASS",         57,  68),  # Net mass (kg)
    ("SUPP_UNIT",        69,  80),  # Supplementary unit
    ("SUPPRESSION",      81,  81),  # Suppression indicator
    ("FLOW",             82,  84),  # Import/Export flow
    ("REC_TYPE",         85,  85),  # Record type flag
]

In [5]:
# Open the file and read just the first line
with open(input_path, "r", encoding="utf-8") as f:
    first_line = f.readline().rstrip("\n\r")

# Show how long the line is
len(first_line)

85

In [6]:
first_line

'20260212026080101210000150001FR007DOV     600000000070570000000015000000000000030exp0'

In [7]:
# Example: manually extract a few fields from the first line

# PERREF: characters 1–6  -> Python slice [0:6]
PERREF = first_line[0:6]

# TYPE: character 7  -> Python slice [6:7]
TYPE = first_line[6:7]

# COMCODE: characters 14–21 -> Python slice [13:21]
COMCODE = first_line[13:21]

# STAT_VALUE: characters 45–56 -> Python slice [44:56]
STAT_VALUE = first_line[44:56]

PERREF, TYPE, COMCODE, STAT_VALUE

('202602', '1', '01012100', '000000007057')

In [8]:
line = first_line  # reuse the line we already read

row = {}

for name, start, end in FIELDS:
    # Convert 1-based inclusive positions to 0-based Python slices
    start0 = start - 1
    end0 = end  # Python slice end is exclusive

    # Extract the substring
    value = line[start0:end0]

    row[name] = value

row

{'PERREF': '202602',
 'TYPE': '1',
 'MONTHAC': '202608',
 'COMCODE': '01012100',
 'SITC': '00150',
 'COD_SEQ': '001',
 'COD_ALPHA': 'FR',
 'PORT_SEQ': '007',
 'PORT_ALPHA': 'DOV',
 'COO_SEQ': '   ',
 'COO_ALPHA': '  ',
 'MODE_OF_TRANSPORT': '60',
 'STAT_VALUE': '000000007057',
 'NET_MASS': '000000001500',
 'SUPP_UNIT': '000000000003',
 'SUPPRESSION': '0',
 'FLOW': 'exp',
 'REC_TYPE': '0'}

In [9]:
# Build the list of column names in order from FIELDS
fieldnames = [name for name, _, _ in FIELDS]
fieldnames

['PERREF',
 'TYPE',
 'MONTHAC',
 'COMCODE',
 'SITC',
 'COD_SEQ',
 'COD_ALPHA',
 'PORT_SEQ',
 'PORT_ALPHA',
 'COO_SEQ',
 'COO_ALPHA',
 'MODE_OF_TRANSPORT',
 'STAT_VALUE',
 'NET_MASS',
 'SUPP_UNIT',
 'SUPPRESSION',
 'FLOW',
 'REC_TYPE']

In [11]:
# alternative way to build the list of column names in order from FIELDS

fieldnames = []

for name, start, end in FIELDS:
    # We only use `name`; start and end are ignored here
    fieldnames.append(name)

fieldnames

['PERREF',
 'TYPE',
 'MONTHAC',
 'COMCODE',
 'SITC',
 'COD_SEQ',
 'COD_ALPHA',
 'PORT_SEQ',
 'PORT_ALPHA',
 'COO_SEQ',
 'COO_ALPHA',
 'MODE_OF_TRANSPORT',
 'STAT_VALUE',
 'NET_MASS',
 'SUPP_UNIT',
 'SUPPRESSION',
 'FLOW',
 'REC_TYPE']

In [12]:

start_time = time.time()

with open(input_path, "r", encoding="utf-8") as f_in, \
     open(output_path, "w", encoding="utf-8", newline="") as f_out:

    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()

    for line_num, line in enumerate(f_in, start=1):
        line = line.rstrip("\n\r")

        if not line.strip():
            # Skip empty lines
            continue

        # (Optional) you could check length here if you like:
        # if len(line) < 85:
        #     print(f"Short line at {line_num}: {len(line)}")

        row = {}
        for name, start, end in FIELDS:
            start0 = start - 1
            end0 = end
            value = line[start0:end0]
            row[name] = value

        writer.writerow(row)

end_time = time.time()
elapsed_seconds = end_time - start_time

print(f"Done. Wrote {output_path}")
print(f"Time taken: {elapsed_seconds:.2f} seconds")

Done. Wrote C:\Users\tcolw\Documents\local_coding_projs\for_fun_weekly_projects\12_py_pipeline\BDSexp2602_parsed.csv
Time taken: 4.87 seconds


In [14]:
import pandas as pd

df = pd.read_csv(output_path)
df.head()

C:\Users\tcolw\AppData\Local\Temp\ipykernel_21500\600817468.py:3: DtypeWarning: Columns (0: COMCODE, 1: SITC, 2: MODE_OF_TRANSPORT, 3: NET_MASS, 4: SUPP_UNIT) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(output_path)


,PERREF,TYPE,MONTHAC,COMCODE,SITC,COD_SEQ,COD_ALPHA,PORT_SEQ,PORT_ALPHA,COO_SEQ,COO_ALPHA,MODE_OF_TRANSPORT,STAT_VALUE,NET_MASS,SUPP_UNIT,SUPPRESSION,FLOW,REC_TYPE
0,202602,1,202608,01012100,00150,1,FR,7,DOV,,,60,7057,1500,3,0,exp,0
1,202602,1,202608,01012100,00150,1,FR,28,PTM,,,60,181589,12000,9,0,exp,0
2,202602,1,202608,01012100,00150,1,FR,104,HLD,,,60,65277,2000,4,0,exp,0
3,202602,1,202608,01012100,00150,1,FR,381,DEU,,,60,1942641,5000,10,0,exp,0
4,202602,1,202608,01012100,00150,3,NL,7,DOV,,,60,5379,1000,2,0,exp,0
